In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/retail_store_sales.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage

In [ ]:
profile = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing %": ((df.isnull().sum() / len(df)) * 100).round(2).values,
    "Unique Values": df.nunique().values
})

profile

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

numeric_columns

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns

categorical_columns

In [ ]:
df.describe(include="all")

In [ ]:
df[numeric_columns].min()

In [ ]:
df[numeric_columns].max()

In [ ]:
profile.to_json("../reports/profiling_report.json", orient="records", indent=4)

In [ ]:
import re

In [ ]:
def detect_semantic_meaning(column_name):
    name = column_name.lower().strip()

    if "email" in name or "mail" in name:
        return "email"

    elif "phone" in name or "mobile" in name or "contact" in name:
        return "phone"

    elif "date" in name or "time" in name:
        return "date"

    elif "name" in name:
        return "name"

    elif "id" in name or name.endswith("_id"):
        return "ID"

    elif "address" in name:
        return "address"

    else:
        return "unknown"

In [ ]:
for column in df.columns:
    print(column, "→", detect_semantic_meaning(column))

In [ ]:
profile["Semantic Meaning"] = [
    detect_semantic_meaning(column)
    for column in df.columns
]

profile

In [ ]:
for column in df.columns:
    print(column, "→", detect_semantic_meaning(column))

In [ ]:
profile["Semantic Meaning"] = [
    detect_semantic_meaning(column)
    for column in df.columns
]

profile

In [ ]:
def detect_mixed_type(column):
    value_types = column.dropna().map(type).nunique()
    return value_types > 1

In [ ]:
mixed_type_results = {}

for column in df.columns:
    mixed_type_results[column] = detect_mixed_type(df[column])

mixed_type_results

In [ ]:
profile["Mixed Type"] = [
    detect_mixed_type(df[column])
    for column in df.columns
]

profile

In [ ]:
def check_type_consistency(column):
    if column.dtype == "object":
        numeric_values = pd.to_numeric(column, errors="coerce")
        
        non_missing = column.notna().sum()
        numeric_count = numeric_values.notna().sum()
        
        if non_missing > 0 and numeric_count == non_missing:
            return "Looks numeric but stored as text"
    
    return "Consistent"

In [ ]:
type_consistency = {}

for column in df.columns:
    type_consistency[column] = check_type_consistency(df[column])

type_consistency

In [ ]:
profile["Type Consistency"] = [
    type_consistency[column]
    for column in df.columns
]

profile

In [ ]:
profile.to_json(
    "../reports/profiling_report.json",
    orient="records",
    indent=4
)

In [ ]:
df.isnull().sum()

In [ ]:
missing_matrix = df.isnull()
missing_matrix.head()

In [ ]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isnull().sum().values,
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2).values
})

missing_summary


In [ ]:
df.nunique()

In [ ]:
cardinality = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": df.nunique().values,
    "Unique Percentage": (
        df.nunique() / len(df) * 100
    ).round(2).values
})

cardinality

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()

numeric_columns

In [ ]:
correlation_matrix = df[numeric_columns].corr()

correlation_matrix

In [ ]:
df[numeric_columns].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Value Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.tight_layout()

plt.savefig("../visualizations/missing_heatmap.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.tight_layout()

plt.savefig("../visualizations/correlation_heatmap.png")
plt.show()

In [ ]:
for column in numeric_columns:
    plt.figure(figsize=(8, 5))
    
    sns.histplot(
        df[column].dropna(),
        kde=True
    )
    
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    
    safe_name = column.replace(" ", "_").lower()
    
    plt.savefig(
        f"../visualizations/{safe_name}_distribution.png"
    )
    
    plt.show()

In [ ]:
outlier_summary = []

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]
    
    outlier_summary.append({
        "Column": column,
        "Outlier Count": len(outliers),
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

In [ ]:
for column in numeric_columns:
    plt.figure(figsize=(8, 5))
    
    sns.boxplot(
        x=df[column].dropna()
    )
    
    plt.title(f"Outlier Distribution - {column}")
    plt.xlabel(column)
    plt.tight_layout()
    
    safe_name = column.replace(" ", "_").lower()
    
    plt.savefig(
        f"../visualizations/{safe_name}_outliers.png"
    )
    
    plt.show()

In [ ]:
import re

def detect_pii(column_name, series):
    name = column_name.lower().strip()

    # Check column name
    pii_keywords = {
        "email": "Possible email information",
        "phone": "Possible phone information",
        "mobile": "Possible phone information",
        "contact": "Possible contact information",
        "name": "Possible personal name",
        "address": "Possible address information",
        "customer": "Possible customer information",
        "user": "Possible user information"
    }

    for keyword, reason in pii_keywords.items():
        if keyword in name:
            return True, reason

    # Check values for email pattern
    if series.dtype == "object":
        sample = series.dropna().astype(str).head(100)

        email_matches = sample.str.match(
            r"^[\w\.-]+@[\w\.-]+\.\w+$"
        ).sum()

        if len(sample) > 0 and email_matches / len(sample) > 0.5:
            return True, "Values look like email addresses"

    return False, "No obvious PII detected"

In [ ]:
for column in df.columns:
    is_pii, reason = detect_pii(
        column,
        df[column]
    )
    
    print(column, "→", is_pii, "→", reason)

In [ ]:
pii_results = []

for column in df.columns:
    is_pii, reason = detect_pii(
        column,
        df[column]
    )
    
    pii_results.append({
        "Potential PII": is_pii,
        "PII Reason": reason
    })

profile["Potential PII"] = [
    item["Potential PII"]
    for item in pii_results
]

profile["PII Reason"] = [
    item["PII Reason"]
    for item in pii_results
]

In [ ]:
def detect_suspicious_column(column_name, series):
    reasons = []

    missing_percentage = series.isnull().mean() * 100
    unique_percentage = series.nunique() / len(series) * 100

    # High missing values
    if missing_percentage > 50:
        reasons.append("High missing values")

    # Almost all values are unique
    if unique_percentage > 95:
        reasons.append("Very high cardinality")

    # Only one unique value
    if series.nunique() <= 1:
        reasons.append("Constant column")

    # Possible ID column
    name = column_name.lower()

    if "id" in name:
        reasons.append("Possible identifier column")

    if reasons:
        return True, "; ".join(reasons)

    return False, "No major issue detected"

In [ ]:
for column in df.columns:
    suspicious, reason = detect_suspicious_column(
        column,
        df[column]
    )
    
    print(column, "→", suspicious, "→", reason)

In [ ]:
suspicious_results = []

for column in df.columns:
    suspicious, reason = detect_suspicious_column(
        column,
        df[column]
    )
    
    suspicious_results.append({
        "Suspicious": suspicious,
        "Suspicious Reason": reason
    })

profile["Suspicious"] = [
    item["Suspicious"]
    for item in suspicious_results
]

profile["Suspicious Reason"] = [
    item["Suspicious Reason"]
    for item in suspicious_results
]

In [ ]:
def check_format_consistency(column_name, series):
    values = series.dropna().astype(str).str.strip()

    if len(values) == 0:
        return "No data"

    name = column_name.lower()

    # Email format check
    if "email" in name or "mail" in name:
        email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        valid = values.str.match(email_pattern).sum()

        if valid / len(values) < 0.9:
            return "Inconsistent email format"
        else:
            return "Consistent"

    # Date format check
    if "date" in name or "time" in name:
        parsed_dates = pd.to_datetime(
            values,
            errors="coerce"
        )

        invalid = parsed_dates.isna().sum()

        if invalid > 0:
            return "Inconsistent date format"
        else:
            return "Consistent"

    # Numeric stored as text
    numeric_values = pd.to_numeric(
        values,
        errors="coerce"
    )

    if numeric_values.notna().sum() / len(values) > 0.9:
        return "Numeric values stored as text"

    # Check unnecessary spaces
    if (values != series.dropna().astype(str)).any():
        return "Inconsistent spaces"

    # Check text capitalization
    if series.dtype == "object":
        unique_values = values.unique()

        lowercase_values = set(
            value.lower() for value in unique_values
        )

        if len(lowercase_values) < len(unique_values):
            return "Possible inconsistent capitalization"

    return "Consistent"

In [ ]:
for column in df.columns:
    result = check_format_consistency(
        column,
        df[column]
    )
    
    print(column, "→", result)

In [ ]:
format_results = []

for column in df.columns:
    result = check_format_consistency(
        column,
        df[column]
    )
    
    format_results.append(result)

profile["Format Consistency"] = format_results

In [ ]:
profile

In [ ]:
profile.to_json(
    "../reports/profiling_report.json",
    orient="records",
    indent=4
)

In [ ]:
import json

# Dataset-level information
dataset_summary = {
    "file_name": "retail_store_sales.csv",
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns
}

# Column-level profiling
column_profiles = []

for column in df.columns:

    series = df[column]

    # Missing values
    missing_count = int(series.isnull().sum())
    missing_percentage = round(
        (missing_count / len(df)) * 100,
        2
    )

    # Unique values
    unique_count = int(series.nunique())

    # Semantic meaning
    semantic_meaning = detect_semantic_meaning(column)

    # Mixed type
    mixed_type = detect_mixed_type(series)

    # Type consistency
    type_consistency = check_type_consistency(series)

    # Format consistency
    format_consistency = check_format_consistency(
        column,
        series
    )

    # PII detection
    potential_pii, pii_reason = detect_pii(
        column,
        series
    )

    # Suspicious column detection
    suspicious, suspicious_reason = detect_suspicious_column(
        column,
        series
    )

    column_profiles.append({
        "column_name": column,
        "data_type": str(series.dtype),
        "semantic_meaning": semantic_meaning,
        "missing_values": missing_count,
        "missing_percentage": missing_percentage,
        "unique_values": unique_count,
        "mixed_type": mixed_type,
        "type_consistency": type_consistency,
        "format_consistency": format_consistency,
        "potential_pii": potential_pii,
        "pii_reason": pii_reason,
        "suspicious": suspicious,
        "suspicious_reason": suspicious_reason
    })

In [ ]:
outlier_profiles = []

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = int(
        (
            (df[column] < lower_bound) |
            (df[column] > upper_bound)
        ).sum()
    )

    outlier_profiles.append({
        "column_name": column,
        "outlier_count": outlier_count,
        "lower_bound": float(lower_bound),
        "upper_bound": float(upper_bound)
    })

In [ ]:
final_report = {
    "dataset_summary": dataset_summary,
    "column_profiles": column_profiles,
    "outlier_summary": outlier_profiles
}

In [ ]:
final_report

In [ ]:
dataset_summary = {
    "file_name": "retail_store_sales.csv",
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "numeric_columns": list(numeric_columns),
    "categorical_columns": list(categorical_columns)
}

In [ ]:
list(numeric_columns)

In [ ]:
list(categorical_columns)

In [ ]:
final_report = {
    "dataset_summary": dataset_summary,
    "column_profiles": column_profiles,
    "outlier_summary": outlier_profiles
}

In [ ]:
with open(
    "../reports/profiling_report.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_report,
        file,
        indent=4
    )

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import os
os.listdir("../src")

In [ ]:
from profiling import (
    detect_semantic_meaning,
    detect_mixed_type,
    check_type_consistency,
    check_format_consistency,
    detect_pii,
    detect_suspicious_column
)

In [ ]:
detect_semantic_meaning("Customer Email")

In [ ]:
detect_semantic_meaning("Customer ID")

In [ ]:
df.columns

In [ ]:
df.columns

In [ ]:
detect_mixed_type(df["Transaction ID"])

In [ ]:
check_type_consistency(df["Transaction ID"])

In [ ]:
detect_pii(
    "Customer ID",
    df["Customer ID"]
)

In [ ]:
detect_suspicious_column(
    "Customer ID",
    df["Customer ID"]
)

In [ ]:
import sys
import os

sys.path.append("../api")

In [ ]:
from profiling_api import generate_profile

In [ ]:
report = generate_profile(
    "../data/retail_store_sales.csv",
    "../reports/api_profiling_report.json"
)

In [ ]:
report["dataset_summary"]

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from cleaning import clean_dataset

In [ ]:
cleaned_df, duplicates_removed = clean_dataset(
    "../data/retail_store_sales.csv",
    "../data/cleaned_data.csv"
)

In [ ]:
cleaned_df.shape

In [ ]:
duplicates_removed

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_report = clean_dataset(
    "../data/retail_store_sales.csv",
    "../data/cleaned_data.csv",
    "../reports/cleaning_report.json"
)

In [ ]:
import pandas as pd

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_report = clean_dataset(
    "../data/retail_store_sales.csv",
    "../data/cleaned_data.csv",
    "../reports/cleaning_report.json"
)

In [ ]:
cleaning_report

In [ ]:
cleaned_df.isnull().sum()

In [ ]:
cleaned_df.duplicated().sum()

In [ ]:
cleaned_df.head()

In [ ]:
cleaned_df.shape

In [ ]:
from validation import validate_dataset

In [ ]:
validation_report = validate_dataset(
    "../data/cleaned_data.csv",
    "../reports/validation_report.json"
)

In [ ]:
validation_report["overall_status"]

In [ ]:
validation_report

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from validation import validate_dataset

In [ ]:
validation_report = validate_dataset(
    "../data/cleaned_data.csv",
    "../reports/validation_report.json"
)

In [ ]:
validation_report["overall_status"]

In [ ]:
validation_report["value_range_check"]

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from pipeline import run_pipeline

In [ ]:
result = run_pipeline(
    "../data/retail_store_sales.csv"
)

In [ ]:
import sys

sys.path.append("../src")

from schema_inference import infer_schema

In [ ]:
import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

schema = infer_schema(df)

schema

In [ ]:
import sys

sys.path.append("../src")

from quality_scoring import calculate_quality_score

In [ ]:
import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

before_score = calculate_quality_score(df)

print("Quality Score Before Cleaning:", before_score)

In [ ]:
import sys

sys.path.append("../src")

from imputation import statistical_imputation

In [ ]:
import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

print("Missing values BEFORE:")
print(df.isnull().sum())

In [ ]:
cleaned_df = statistical_imputation(df)

print("Missing values AFTER:")
print(cleaned_df.isnull().sum())

In [ ]:
import sys

sys.path.append("../src")

from imputation import statistical_imputation

In [ ]:
import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

print("Missing values BEFORE:")
print(df.isnull().sum())

In [ ]:
cleaned_df = statistical_imputation(df)

print("Missing values AFTER:")
print(cleaned_df.isnull().sum())

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import sys

sys.path.append("../src")

from imputation import knn_imputation

In [ ]:
import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

print("Missing values BEFORE:")
print(df.isnull().sum())

In [ ]:
knn_df = knn_imputation(df)

print("Missing values AFTER KNN:")
print(knn_df.isnull().sum())

In [ ]:
print("Original shape:", df.shape)
print("KNN result shape:", knn_df.shape)

In [ ]:
import sys

sys.path.append("../src")

from imputation import regression_imputation

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/retail_store_sales.csv"
)

print("Missing values BEFORE:")
print(df.isnull().sum())

In [ ]:
regression_df = regression_imputation(df)

print("Missing values AFTER Regression:")
print(regression_df.isnull().sum())

In [ ]:
import sys

sys.path.append("../src")

from imputation import iterative_imputation

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/retail_store_sales.csv"
)

print("Missing values BEFORE:")
print(df.isnull().sum())

In [ ]:
iterative_df = iterative_imputation(df)

print("Missing values AFTER Iterative Imputation:")
print(iterative_df.isnull().sum())

In [ ]:
print("Original shape:", df.shape)
print("Iterative result shape:", iterative_df.shape)

In [ ]:
import sys

sys.path.append("../src")

from duplicate_detection import (
    find_exact_duplicates,
    find_fuzzy_duplicates
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/retail_store_sales.csv"
)

exact_duplicates = find_exact_duplicates(df)

print(
    "Exact duplicate rows:",
    exact_duplicates
)

In [ ]:
fuzzy_duplicates = find_fuzzy_duplicates(
    df,
    threshold=90
)

print(
    "Fuzzy duplicate pairs:",
    len(fuzzy_duplicates)
)

In [ ]:
fuzzy_duplicates[:10]

In [ ]:
import sys

sys.path.append("../src")

from normalization import normalize_dataset

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/retail_store_sales.csv"
)

print("BEFORE:")
print(df.dtypes)

In [ ]:
normalized_df = normalize_dataset(df)

print("AFTER:")
print(normalized_df.dtypes)

In [ ]:
normalized_df.head()

In [ ]:
print(
    "Rows:",
    normalized_df.shape[0]
)

print(
    "Columns:",
    normalized_df.shape[1]
)

In [ ]:
import sys

sys.path.append("../src")

from transformation import transform_dataset

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/retail_store_sales.csv"
)

In [ ]:
print("Before transformation:")
print(df.shape)
print(df.dtypes)

In [ ]:
transformed_df = transform_dataset(df)

In [ ]:
print("After transformation:")
print(transformed_df.shape)
print(transformed_df.dtypes)

In [ ]:
transformed_df.head()

In [ ]:
import sys
sys.path.append("../src")

from quality_scoring import calculate_quality_score, get_quality_metrics

import pandas as pd

df = pd.read_csv("../data/retail_store_sales.csv")

before_score = calculate_quality_score(df)

print("Quality Score Before Cleaning:", before_score)

metrics = get_quality_metrics(df)

print(metrics)

In [ ]:
from quality_scoring import compare_quality_scores

before = 70
after = 90

result = compare_quality_scores(before, after)

print(result)

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_log = clean_dataset(
    "../data/retail_store_sales.csv",
    "../reports/cleaned_data.csv",
    "../reports/cleaning_log.json"
)

In [ ]:
print("Cleaning completed!")
print("Original rows:", cleaning_log["original_rows"])
print("Cleaned rows:", cleaning_log["cleaned_rows"])
print("Duplicates removed:", cleaning_log["duplicates_removed"])

In [ ]:
print(cleaning_log["quality_comparison"])

In [ ]:
import sys
sys.path.append("../src")

from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_log = clean_dataset(
    "../data/retail_store_sales.csv",
    "../reports/cleaned_data.csv",
    "../reports/cleaning_log.json"
)

print("Cleaning completed!")

In [ ]:
print(cleaning_log["quality_comparison"])

In [ ]:
import sys
sys.path.append("../src")

from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_log = clean_dataset(
    "../data/retail_store_sales.csv",
    "../reports/cleaned_data.csv",
    "../reports/cleaning_log.json"
)

print("Cleaning completed!")

In [ ]:
print("Missing before:",
      cleaning_log["missing_values_before"])

print("Missing after:",
      cleaning_log["missing_values_after"])

print("Values imputed:",
      cleaning_log["values_imputed"])

print("Duplicates removed:",
      cleaning_log["duplicates_removed"])

print("Quality comparison:")
print(cleaning_log["quality_comparison"])

In [ ]:
import sys
sys.path.append("../src")

from cleaning import clean_dataset

In [ ]:
cleaned_df, cleaning_log = clean_dataset(
    "../data/retail_store_sales.csv",
    "../reports/cleaned_data.csv",
    "../reports/cleaning_log.json"
)

print("Cleaning completed!")

In [ ]:
print("Missing before:",
      cleaning_log["missing_values_before"])

print("Missing after:",
      cleaning_log["missing_values_after"])

print("Values imputed:",
      cleaning_log["values_imputed"])

print("Duplicates removed:",
      cleaning_log["duplicates_removed"])

print("Quality comparison:")
print(cleaning_log["quality_comparison"])

In [ ]:
import sys

sys.path.append("../api")

from cleaning_api import run_cleaning_api

In [ ]:
result = run_cleaning_api(
    "../data/retail_store_sales.csv",
    "../reports/cleaned_data.csv",
    "../reports/cleaning_log.json"
)

In [ ]:
print(result)

In [ ]:
import sys

sys.path.append("../src")

from validation_rules import (
    validate_dataset_structure,
    validate_missing_values
)

import pandas as pd

In [ ]:
df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

In [ ]:
structure_result = validate_dataset_structure(df)

print(structure_result)

In [ ]:
missing_result = validate_missing_values(df)

print(missing_result[:5])

In [ ]:
import sys

sys.path.append("../src")

from validation_rules import (
    validate_dataset_structure,
    validate_missing_values,
    validate_email_column,
    validate_phone_column,
    validate_postcode_column
)

import pandas as pd

In [ ]:
df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.columns.tolist())

In [ ]:
print(validate_dataset_structure(df))

print(validate_missing_values(df)[:5])

In [ ]:
test_df = pd.DataFrame({
    "Email": [
        "john@gmail.com",
        "wrong-email",
        "alice@yahoo.com"
    ],
    "Phone": [
        "9876543210",
        "12345",
        "9123456789"
    ],
    "Postcode": [
        "500001",
        "123",
        "600001"
    ]
})

In [ ]:
print(validate_email_column(test_df, "Email"))
print(validate_phone_column(test_df, "Phone"))
print(validate_postcode_column(test_df, "Postcode"))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.columns.tolist())

In [ ]:
print(df.select_dtypes(include="number").columns.tolist())

In [ ]:
from validation_rules import validate_numeric_range

result = validate_numeric_range(
    df,
    "Price",
    minimum=0,
    maximum=100000
)

print(result[:10])

In [ ]:
import sys
sys.path.append("../src")

from validation_rules import validate_numeric_range

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.columns.tolist())

In [ ]:
print(df.select_dtypes(include="number").columns.tolist())

In [ ]:
result = validate_numeric_range(
    df,
    "Price",
    minimum=0,
    maximum=100000
)

print(result[:10])

In [ ]:
test_df = pd.DataFrame({
    "Price": [100, 250, -50, 500]
})

result = validate_numeric_range(
    test_df,
    "Price",
    minimum=0,
    maximum=1000
)

print(result)

In [ ]:
import sys
sys.path.append("../src")

from validation_rules import validate_categorical_consistency

In [ ]:
import pandas as pd

test_df = pd.DataFrame({
    "Payment": [
        "Cash",
        "UPI",
        "Credit Card",
        "cashh",
        "UPI"
    ]
})

result = validate_categorical_consistency(
    test_df,
    "Payment",
    ["Cash", "UPI", "Credit Card"]
)

print(result)

In [ ]:
import sys

sys.path.append("../src")

from anomaly_detection import (
    detect_isolation_forest_anomalies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.shape)

In [ ]:
result = detect_isolation_forest_anomalies(df)

print("Status:", result["status"])
print("Method:", result["method"])
print("Total rows:", result["total_rows"])
print("Anomalies:", result["anomaly_count"])
print(
    "Anomaly percentage:",
    result["anomaly_percentage"]
)

In [ ]:
anomalies = [
    row for row in result["results"]
    if row["status"] == "anomaly"
]

print(anomalies[:10])

In [ ]:
import sys

sys.path.append("../src")

from anomaly_detection import (
    detect_lof_anomalies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.shape)

In [ ]:
result_lof = detect_lof_anomalies(df)

print("Status:", result_lof["status"])
print("Method:", result_lof["method"])
print("Total rows:", result_lof["total_rows"])
print("Anomalies:", result_lof["anomaly_count"])
print(
    "Anomaly percentage:",
    result_lof["anomaly_percentage"]
)

In [ ]:
import sys

sys.path.append("../src")

from anomaly_detection import (
    detect_isolation_forest_anomalies,
    detect_lof_anomalies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.shape)

In [ ]:
result_if = detect_isolation_forest_anomalies(df)

print("Isolation Forest")
print("Status:", result_if["status"])
print("Anomalies:", result_if["anomaly_count"])
print(
    "Anomaly percentage:",
    result_if["anomaly_percentage"]
)

In [ ]:
result_lof = detect_lof_anomalies(df)

print("LOF")
print("Status:", result_lof["status"])
print("Anomalies:", result_lof["anomaly_count"])
print(
    "Anomaly percentage:",
    result_lof["anomaly_percentage"]
)

In [ ]:
import sys

sys.path.append("../src")

from anomaly_detection import (
    detect_isolation_forest_anomalies,
    detect_lof_anomalies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

In [ ]:
result_lof = detect_lof_anomalies(df)

print("Status:", result_lof["status"])
print("Method:", result_lof["method"])
print("Unique rows analyzed:", result_lof["unique_rows_analyzed"])
print(
    "Duplicate numeric rows excluded:",
    result_lof["duplicate_numeric_rows_excluded"]
)
print("Anomalies:", result_lof["anomaly_count"])
print(
    "Anomaly percentage:",
    result_lof["anomaly_percentage"]
)

In [ ]:
import sys

sys.path.append("../src")

from anomaly_detection import (
    detect_autoencoder_anomalies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.shape)

In [ ]:
result_autoencoder = detect_autoencoder_anomalies(
    df
)

print(
    "Status:",
    result_autoencoder["status"]
)

print(
    "Method:",
    result_autoencoder["method"]
)

print(
    "Total rows:",
    result_autoencoder["total_rows"]
)

print(
    "Anomalies:",
    result_autoencoder["anomaly_count"]
)

print(
    "Anomaly percentage:",
    result_autoencoder["anomaly_percentage"]
)

In [ ]:
autoencoder_anomalies = [
    row
    for row in result_autoencoder["results"]
    if row["status"] == "anomaly"
]

print(
    autoencoder_anomalies[:10]
)

In [ ]:
import sys

sys.path.append("../src")

from column_classifier import (
    classify_columns,
    detect_semantic_inconsistencies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

print(df.columns.tolist())

In [ ]:
import sys

sys.path.append("../src")

from column_classifier import (
    classify_columns,
    detect_semantic_inconsistencies
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

In [ ]:
classification_results = classify_columns(df)

for result in classification_results:
    print(result)

In [ ]:
inconsistencies = detect_semantic_inconsistencies(df)

print("Number of inconsistencies:")
print(len(inconsistencies))

print("\nExamples:")

for item in inconsistencies[:10]:
    print(item)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from suspicious_patterns import (
    detect_suspicious_patterns
)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

In [ ]:
suspicious_results = detect_suspicious_patterns(df)

print(
    "Total suspicious findings:",
    len(suspicious_results)
)

In [ ]:
for result in suspicious_results[:10]:
    print(result)

In [ ]:
from collections import Counter

rule_counts = Counter(
    result["rule"]
    for result in suspicious_results
)

print(rule_counts)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from data_drift import detect_data_drift

In [ ]:
import pandas as pd

reference_df = pd.read_csv(
    "../reports/cleaned_data.csv"
)

current_df = reference_df.copy()

In [ ]:
drift_results = detect_data_drift(
    reference_df,
    current_df
)

print("Columns checked:", len(drift_results))

for result in drift_results:
    print(result)

In [ ]:
print(current_df.columns.tolist())

In [ ]:
current_df["Total Spent"] = (
    current_df["Total Spent"] * 3
)

In [ ]:
drift_results = detect_data_drift(
    reference_df,
    current_df
)

for result in drift_results:
    if result["column"] == "Total Spent":
        print(result)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from severity_scoring import (
    generate_severity_report
)

In [ ]:
test_results = [
    {
        "column": "Price",
        "rule": "numeric_range",
        "status": "failed",
        "severity": "high",
        "message": "Negative price detected."
    },
    {
        "column": "Payment",
        "rule": "categorical_consistency",
        "status": "failed",
        "severity": "medium",
        "message": "Unexpected category."
    },
    {
        "column": "Quantity",
        "rule": "zero_value",
        "status": "warning",
        "severity": "low",
        "message": "Zero quantity detected."
    },
    {
        "column": "Category",
        "rule": "categorical_consistency",
        "status": "passed",
        "severity": "none",
        "message": "Valid category."
    }
]

In [ ]:
severity_report = generate_severity_report(
    test_results
)

print(severity_report)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from validation_pipeline import (
    generate_validation_report
)

In [ ]:
report = generate_validation_report(
    cleaned_file="../reports/cleaned_data.csv",
    report_file="../reports/validation_report.json"
)

In [ ]:
print("Validation report generated successfully.")

In [ ]:
print(
    report["severity_and_health"]
)

In [ ]:
import os

print(
    os.path.exists(
        "../reports/validation_report.json"
    )
)

In [ ]:
import sys

sys.path.append("../api")

In [ ]:
from validation_api import run_validation_api

In [ ]:
result = run_validation_api(
    cleaned_file="../reports/cleaned_data.csv",
    report_file="../reports/validation_report.json"
)

print(result)

In [ ]:
import os

print(
    os.path.exists(
        "../reports/validation_report.json"
    )
)


In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from model_evaluation import (
    evaluate_anomaly_predictions,
    create_demo_labels
)

In [ ]:
actual_labels = create_demo_labels(
    total_rows=20,
    anomaly_indexes=[3, 7, 12]
)

print(actual_labels)

In [ ]:
predicted_labels = [
    0, 0, 0, 1, 0,
    0, 1, 1, 0, 0,
    0, 0, 1, 0, 0,
    0, 0, 0, 0, 0
]

In [ ]:
evaluation = evaluate_anomaly_predictions(
    actual_labels,
    predicted_labels
)

print(evaluation)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from model_evaluation import (
    calculate_roc_curve,
    plot_roc_curve
)

In [ ]:
anomaly_scores = [
    0.10,
    0.20,
    0.15,
    0.90,
    0.30,
    0.25,
    0.80,
    0.95,
    0.20,
    0.10,
    0.15,
    0.25,
    0.85,
    0.20,
    0.15,
    0.10,
    0.20,
    0.15,
    0.10,
    0.20
]

In [ ]:
anomaly_scores = [
    0.10,
    0.20,
    0.15,
    0.90,
    0.30,
    0.25,
    0.80,
    0.95,
    0.20,
    0.10,
    0.15,
    0.25,
    0.85,
    0.20,
    0.15,
    0.10,
    0.20,
    0.15,
    0.10,
    0.20
]

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from model_evaluation import (
    create_demo_labels,
    evaluate_anomaly_predictions,
    calculate_roc_curve,
    plot_roc_curve
)

In [ ]:
actual_labels = create_demo_labels(
    total_rows=20,
    anomaly_indexes=[3, 7, 12]
)

print(actual_labels)

In [ ]:
anomaly_scores = [
    0.10,
    0.20,
    0.15,
    0.90,
    0.30,
    0.25,
    0.80,
    0.95,
    0.20,
    0.10,
    0.15,
    0.25,
    0.85,
    0.20,
    0.15,
    0.10,
    0.20,
    0.15,
    0.10,
    0.20
]

In [ ]:
roc_result = calculate_roc_curve(
    actual_labels,
    anomaly_scores
)

print(roc_result)

In [ ]:
plot_roc_curve(
    actual_labels,
    anomaly_scores
)

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from validation_pipeline import (
    generate_validation_report
)

In [ ]:
report = generate_validation_report(
    cleaned_file="../reports/cleaned_data.csv",
    report_file="../reports/validation_report.json"
)

print("Validation report updated successfully.")

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from validation_pipeline import generate_validation_report

In [ ]:
report = generate_validation_report(
    cleaned_file="../reports/cleaned_data.csv",
    report_file="../reports/validation_report.json"
)

print("Validation report updated successfully.")

In [1]:
import sys
sys.path.append("../src")

In [2]:
from validation_pipeline import generate_validation_report

In [1]:
import sys
sys.path.append("../src")

In [2]:
from validation_pipeline import generate_validation_report

In [3]:
report = generate_validation_report(
    cleaned_file="../reports/cleaned_data.csv",
    report_file="../reports/validation_report.json"
)

print("Validation report updated successfully.")

Validation report updated successfully.


In [4]:
import os

print(
    os.path.exists(
        "../reports/validation_report.json"
    )
)

True
